# Deep Momentum V2 — research master

This notebook is the entry point you can follow in VS Code and rerun with **Run All**.
It calls small, tested worker files in `src/dmv2/`; the loading rules are coded there.

**Selected design:** research initialization in **2010**, rolling **10 years training / 1 year testing**.
Nominal outcome tests are **2020–2023**. Starting later is a research choice, not a claim that earlier raw history is absent.
The source files still contain their longer histories.

Current Run All performs configuration, fold planning, source checks and one bounded
historical-universe audit, then a bounded monthly-return engineering sample.
Daily prices stay intact for later daily backtesting. A bounded feature stage calculates daily indicators and exact month-end snapshots. Saved full feature and target audits are verified below; full builds remain explicit CLI actions. Section 17 verifies saved results from the separately authorized four-arm RF/XGBoost development pilot (2010–2017 training, 2018–2019 validation). Run All does not fit models.
No market-capitalization input is used. Protected 2024+ financial values remain outside this workflow.


## 1. Environment and project root

Select **Deep Momentum V2 (.venv)** in the VS Code kernel picker. This bootstrap
finds V2 from its root, the `notebooks/` folder or the Finance-Working-Files repository.
It imports the current local worker code explicitly and enables reloads during development.
It does not install packages, log in, download data or open cloud resources.


In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import asdict
from html import escape
from IPython.display import display, HTML, Markdown

# Clear prior run evidence before setup, so failed reruns cannot display or save old results.
settings = catalog = inspection = audit_path = normalization_protocol = None
identity_audit = identity_audit_path = None
research_identity_df = research_identity_summary = None
feature_run = feature_history_df = features_df = feature_coverage_df = feature_catalog_df = feature_artifact_path = None
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None
target_ret_protocol = shared_target_run = target_classes_df = target_diagnostics_df = ret_sample_payoffs = None
full_target_run = full_target_dataset = full_monthly_outcome_dataset = None
full_target_coverage_df = full_target_class_coverage_df = full_ret_coverage_df = None
classifier_pilot_run = classifier_pilot_comparison_df = None
bbby_identity_review = bbby_identity_summary = None

cwd = Path.cwd().resolve()
bases = [cwd, *cwd.parents]
candidates = bases + [p / "Investments_2026" / "deep_momentum_v2" for p in bases]
ROOT = next((p for p in candidates
             if (p / "configs/research_workflow.json").is_file()
             and (p / "src/dmv2/workflow.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the V2 folder in VS Code and restart the notebook kernel.")
SRC = (ROOT / "src").resolve()
for name, module in list(sys.modules.items()):
    if name == "dmv2" or name.startswith("dmv2."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(SRC):
            raise RuntimeError("The kernel has V2 code from another folder. Restart the kernel.")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import dmv2
assert Path(dmv2.__file__).resolve().is_relative_to(SRC)
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")
from dmv2.catalog import SourceCatalog
from dmv2.chronology import build_annual_folds
from dmv2.workflow import (load_workflow, run_inspection, save_inspection,
                            run_monthly_sample, save_monthly_sample)

def show_table(rows, columns):
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>"
                   for r in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Project:", ROOT)
print("Python:", sys.executable)
print("Worker code:", Path(dmv2.__file__).parent)

from dmv2.feature_workflow import run_feature_sample, save_feature_sample
from dmv2.features import predictor_frame, training_slice

from dmv2.workflow import workflow_identity_policy
from dmv2.identity_policy import identity_policy_provenance, qualify_audit_ledger


## 2. One settings file

Edit `configs/research_workflow.json`, then rerun this notebook from the top.
The study start, engineering audit date and bounded monthly sample come from this file.
The current initialization is 2010; the engineering audit date is a separate setting.
`engineering_sample = combined` is a loader demonstration, not a frozen primary modelling universe.
Price sampling is off by default; enabling it reads at most the explicitly configured small ID sample.

The monthly stage is enabled for **January–February 2010 and two requested price IDs**.
Its full monthly cohort ledger is retained. A December 2009 signal anchor is necessary for
January returns; earlier observations are feature warmup, not earlier training outcomes.

In [ ]:
settings = None
settings = load_workflow(ROOT)
print(json.dumps(settings, indent=2))
monthly_controls = settings["monthly_returns"]
display(Markdown(f"Research initialization: **{settings['research_start']}**; "
                 f"engineering snapshot: **{settings['inspection']['as_of']}**; "
                 f"monthly engineering sample: **{monthly_controls['start_month']}–{monthly_controls['end_month']}**, "
                 f"**{monthly_controls['sample_size']} requested price IDs**."))

print(identity_policy_provenance(workflow_identity_policy(settings))["warning"])


### Active normalization and missingness rules

`configs/normalization_protocol.json` records the active engineering contract
under our accepted provisional identity policy. The former proposal and its
2000–2009 strict-identity audit are archived reference evidence.

Request every admitted original member before signal-time statistics. The
paper's 15 inputs share one population with all five exact finite raw momenta,
N≥2 and positive finite sample SD at every horizon. Preserve shared meanMOM
and stdMOM levels over time. The two-ID sample withholds normalization because
it is not an exhaustive cohort request.

For added predictors, fit medians and the retained schema only inside the
relevant training partition; add a fixed missingness indicator for each retained
feature. No extra scaling, clipping, winsorization or full-history feature screen
is used. Never impute a label. Missing added features do not remove otherwise
qualified prediction rows. Added context uses the explicit 80% primary original-cohort
policy (90% sensitivity) and remains withheld without the required original/daily context.


In [ ]:
normalization_protocol = None
normalization_protocol = json.loads((ROOT / "configs/normalization_protocol.json").read_text())
assert settings["features"]["cohort_policy"] == normalization_protocol["paper_normalization"]["policy"]
assert normalization_protocol["chronology"]["initial_outcome_year"] == int(settings["research_start"][:4])
print("Active protocol:", normalization_protocol["protocol_id"])
print("Status:", normalization_protocol["status"])
print("Paper population policy:", normalization_protocol["paper_normalization"]["policy"])
print("Missingness is fitted only on completed, explicitly selected training rows.")
print("No preprocessing fit or model fit is run here.")

assert settings["features"]["context_policy"] == normalization_protocol["context_policy"]["active_worker_policy"]
print("Added-context policy:", normalization_protocol["context_policy"])


## 3. Rolling chronology — calendar plan, not a trained model

These windows refer to **completed outcome years**. A December 2019 signal
predicting January 2020 belongs to the 2020 test; its future label cannot enter
training ending in December 2019. Actual training rows must also have observable labels at the fit cutoff.

The nominal 2010–2019 → 2020 fold needs verified coverage and feature warmup.
Early 2010 inputs use exact pre-signal windows, including pre-2010 prices.
Selecting 2010 does not truncate the raw files or establish 120 complete usable training months.
Rolling classifier history is our chosen adaptation to the paper's expanding history.
The paper's trailing ten-year class-payoff estimator is a separate component.


In [ ]:
training = settings["training"]
folds = build_annual_folds(settings["research_start"], training["last_test_year"],
                           training["train_years"], mode=training["mode"])
show_table([asdict(f) for f in folds],
           ["fold_id", "train_start", "train_end", "label_cutoff", "test_start", "test_end"])
print(len(folds), "nominal annual folds; no features, fits or backtests executed.")
first_fold, last_fold = folds[0], folds[-1]
display(Markdown(f"First fold: training **{first_fold.train_start}–{first_fold.train_end}**, "
                 f"test **{first_fold.test_start}–{first_fold.test_end}**. "
                 f"Final test ends **{last_fold.test_end}**."))


## 4. Local source inventory

The catalog uses manifest-listed files and canonical manifest pins, not directory discovery.
This checks file presence and sizes. It does not repeat the prior whole-file SHA-256
verification or claim complete historical returns. A fresh clone needs your own approved
`configs/data_sources.local.json` and downloaded source releases.


In [ ]:
catalog = None
catalog = SourceCatalog(ROOT)
inventory = catalog.verify(hashes=False)
show_table([asdict(r) for r in inventory],
           ["source_key", "release_id", "object_count", "size_bytes", "hashes_checked"])
print(sum(r.object_count for r in inventory), "pinned local objects present.")


## 5. Bounded historical constituent audit

The default query is one source session, **29 January 2010**, for the S&P 500/400 union.
The date is read from `inspection.as_of`; this is a separate bounded inspection.
It does not scan all price history or build the research-period feature panel. Source storage is arranged
by security/index history, so many metadata rows may be considered for this one-date result;
the configured row/file/byte limits apply and failures never return truncated success.

`historical_reconstruction` respects supplied date rules. The sources do not certify true
historical knowledge-time identity/revision evidence. Current delisting flags and future
quote availability never select the historical cohort.


In [ ]:
inspection = audit_path = None
inspection = run_inspection(catalog, settings)
if inspection is None:
    display(Markdown("Universe inspection is disabled in the settings file."))
else:
    print("Research initialization:", settings["research_start"])
    print("Engineering snapshot:", inspection.snapshot.as_of)
    print(json.dumps(inspection.snapshot.summary, indent=2))
    show_table(inspection.snapshot.table.select(
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"]).to_pylist()[:12],
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"])
    print("Preview only; all", inspection.snapshot.table.num_rows, "constituents stay in the ledger.")


### Read the identity statuses correctly

`needs_review` preserves an upstream caution flag. It does **not** mean the stock is
absent or its price history is necessarily wrong. The prior 2010 sample's 755 flagged
links all had provider IDs and successful resolution; 751 used ticker/company matches.
Some warnings can represent harmless aliases; date-specific conflicts need separate evidence.

The notebook retains all original constituents. Do not train on only the unflagged subset
as though it were the full intended universe. Historical identity validity and terminal-event
accounting remain the next data-quality tasks.


### Historical identity evidence review — reference anchors

This separate metadata worker still checks **2000-01-31, 2005-01-31 and 2009-01-30**.
These fixed diagnostic anchors predate the current 2010 initialization and do not
measure its training coverage. The current engineering snapshot is controlled separately.
 The exact three-date membership filter and shared identity tables have a cumulative ceiling of 20 million candidate rows, 100,000 returned metadata rows, 12 data files and 128 MiB projected compressed bytes. No price or outcome table is read.

Every original member stays in `identity_audit.table` and its private review queue. ISIN aliases are source agreement, not independent provider confirmation. A valid matching declared ISIN query is retrieval-time evidence awaiting raw request provenance. Source identifier dates, even if present, cannot certify undated provider links or publication times. The audit promotes no identities and does not change feature eligibility. See [identity audit method](../docs/IDENTITY_AUDIT.md).


In [ ]:
from dmv2.identity_audit import build_identity_audit, save_identity_audit
identity_audit = identity_audit_path = None
identity_audit = build_identity_audit(catalog)
show_table(identity_audit.summary["snapshots"], [
    "signal_date", "original_members", "original_resolved_members",
    "exact_declared_isin_queries", "dated_identifier_support_rows", "promoted_identities"])
print("Identifier intervals:", identity_audit.interval_profile)
print("Metadata read budget evidence:", identity_audit.provenance["loading_stats"]["total"])
identity_audit_path = save_identity_audit(ROOT, identity_audit)
print("Private identity evidence:", identity_audit_path.relative_to(ROOT))
print("Identity eligibility changed:", identity_audit.summary["eligibility_changed"])


### User-selected provisional identity admission

The original audit remains unchanged. `identity.policy` now permits uniquely linked, resolved provider routes carrying a producer review flag for **provisional research**. Missing/ambiguous routes and same-date provider collisions stay blocked and retain their original rows. `research_identity_df` is a separate admission ledger, never a predictor source.

Original flags are not cleared. Historical identity/PIT/survivorship remain unverified and later research must carry that limitation. Monthly and feature workers consume the same explicit policy. The two-ID smoke selects an unflagged control and a flagged link before prices; unavailable history never triggers a replacement. Full-cohort normalization remains separate from identity admission. See [the recorded decision](../docs/PROVISIONAL_IDENTITY_POLICY.md).


In [ ]:
import pandas as pd
research_identity_df = research_identity_summary = None
admission_rows, research_identity_summary = qualify_audit_ledger(
    identity_audit.table, identity_policy=workflow_identity_policy(settings))
research_identity_df = pd.DataFrame(admission_rows)
show_table(research_identity_summary, ["signal_date", "original_members",
    "identity_eligible_members", "review_flag_accepted_members", "blocked_members"])
print("Admission policy:", workflow_identity_policy(settings))
print("The original identity_audit.table and producer flags remain intact.")


## 6. Optional exact-date price check

Set `inspection.run_price_sample` to `true` for the configured small engineering sample.
This cell summarizes the result. Unqueried members remain `not_requested`; unresolved
identity links remain visible. This does not certify whole-history coverage or returns.


In [ ]:
if inspection is None or inspection.prices is None:
    display(Markdown("This optional one-date quote check is disabled. The separate monthly stage below has its own controls."))
else:
    print(json.dumps(inspection.prices.stats, indent=2))
    print(json.dumps(inspection.quote_coverage.summary, indent=2))


## 7. Private audit artifact and reruns

The complete constituent/exclusion ledger and source/request provenance are saved under
ignored `outputs/notebook_audits/`. The same settings/source/code/runtime pins reuse an identical report.
Worker edits produce a separate fingerprinted artifact; changed evidence under otherwise identical pins raises an error.
The versioned notebook is kept without execution outputs; clear outputs before committing it.


In [ ]:
audit_path = None
if inspection is not None and settings["inspection"]["persist_audit"]:
    audit_path = save_inspection(ROOT, inspection)
    print("Private audit:", audit_path)
else:
    print("Private audit persistence is disabled.")


## 8. Signal-only features before outcomes

The feature worker reads a separate daily history ending at the **latest signal close**.
It never takes `monthly_df` as input. Each earlier snapshot uses only its own past.
`feature_history_df` holds this past-only history; `features_df` contains source-ID/signal
keys and registered predictors. Coverage/identity flags stay in `feature_coverage_df`.

The bank has **728 definitions**: 298 daily technical, 415 S&P 500/400 context and
15 paper monthly momentum/state inputs. Three constants and ten duplicate ranks
remain documented but are excluded by `predictor_frame`. Missing values stay missing.
V2 requires exact complete temporal windows. Added-context statistics require exhaustive
admitted-ID requests and 80% valid original members (90% sensitivity); the paper15 use
the unchanged common-five population. These policies and sample SD(ddof=1) are adaptations.

Daily features are evaluated before selecting exact exchange month ends. The paper
variables retain calendar-month indexing: MOM1 uses month t; MOM3/6/9/12 use
2/5/8/11 completed returns through t-1. Their normalization is at signal time.
The longest abnormal-volume window needs **503 sessions**; observations before the
configured 2010 initialization are warmup rather than earlier training outcomes.

The default two-ID sample cannot support full-universe ranks or paper normalization.
Those outputs stay null with explicit reasons. Market/relative features also stay
null until full dated DAILY membership context is supplied under a measured plan.
No sampled average pretends to be the market.

Features end at signal close t. The next-month return interval is **(t, t+1 month-end]**.
Sharing the signal price as the return's starting anchor does not reveal a future return.
Same-close execution is not assumed. Source revision knowledge, identity continuity and
terminal accounting remain unverified; `knowledge_time_verified` and `model_ready` are false.


In [ ]:
feature_run = feature_history_df = features_df = feature_coverage_df = feature_catalog_df = feature_artifact_path = None
feature_run = run_feature_sample(catalog, settings)
if feature_run is None:
    display(Markdown("Feature engineering is disabled in the settings file."))
else:
    feature_history_df = feature_run.sample.daily_prices.table.to_pandas().copy(deep=True)
    features_df = feature_run.panel.table.to_pandas().copy(deep=True)
    feature_coverage_df = feature_run.panel.coverage.to_pandas().copy(deep=True)
    feature_catalog_df = __import__("pandas").DataFrame(feature_run.panel.specs)
    print(json.dumps(feature_run.panel.summary, indent=2))
    print("Independent feature history request:")
    print(json.dumps(feature_run.sample.loading_provenance["request"], indent=2))
    print("Measured cumulative feature-stage reads:")
    print(json.dumps(feature_run.sample.loading_stats["total"], indent=2))
    assert feature_history_df["trading_date"].max() <= max(feature_run.sample.signal_dates)
    assert "provider_adjusted_return" not in features_df.columns
    assert "return_status" not in features_df.columns


### Inspect stock features and missingness separately

`predictor_frame(feature_run.panel)` is the future X interface. It accepts only
eligible dictionary names and excludes raw momentum diagnostics, outcomes and flags.
It does not fill missing values, fit a scaler or select rows. The display below
is an engineering preview; all original constituents remain in the feature ledger.


In [ ]:
if feature_run is not None:
    preview = features_df.merge(feature_coverage_df[["source_security_id", "signal_date", "feature_status"]],
                                on=["source_security_id", "signal_date"], validate="one_to_one")
    preview = preview.loc[preview["feature_status"] == "partial_features"].head(10)
    show_table(preview.to_dict("records"), ["source_security_id", "signal_date",
        "px_simple_return_20s", "risk_realized_vol_20s", "trend_ma_gap_20s",
        "liq_abnormal_volume_ratio_20s", "zMOM1", "feature_status"])
    print("Dictionary families:")
    display(feature_catalog_df.groupby(["role", "family"], sort=False).size().rename("definitions").to_frame())
    print("Paper raw momentum diagnostics (outside the predictor dictionary):")
    raw = feature_run.panel.diagnostics["paper_raw_momentum"]
    display(raw.loc[raw["MOM1_raw_status"] == "ok"].head(4))
    print("Coverage reasons stay separate from predictors:")
    for name in ("daily", "paper", "context"):
        diagnostic = feature_run.panel.diagnostics[name]
        print(name, "diagnostic rows:", len(diagnostic))
    print("Eligible predictor columns:", predictor_frame(feature_run.panel).shape[1])


In [ ]:
feature_artifact_path = None
if feature_run is not None and settings["features"]["persist"]:
    feature_artifact_path = save_feature_sample(ROOT, feature_run)
    print("Private feature evidence:", feature_artifact_path)
else:
    print("Feature evidence persistence is disabled.")


## 9. Monthly returns with the original historical cohort

Each outcome month uses membership at the **preceding source month end**.
With the current settings, January 2010 uses the December 2009 signal cohort;
February uses the January cohort. The table below derives exact dates from the settings.
A removal during the outcome month remains in that month's ledger. Entrants during
the month enter the next signal cohort, without retroactive admission.

The worker computes `provider_adjusted_return = adj_close[end] / adj_close[signal] - 1`
only at exact consecutive independent XNYS month ends, from completed daily bars and
accepted endpoint quality. No last-observed-price fallback, forward fill, dividend
double counting or zero-return imputation is used. Interior daily gaps are audited
separately from endpoint arithmetic. Unrequested prices are distinct from missing prices.

Source adjustments include splits and dividends, but historical identity continuity,
revision availability, merger stitching and terminal cash flows are not certified.
`accounting_certified`, `knowledge_time_verified` and `daily_backtest_ready` remain false.
These are research return observations, not executable portfolio returns or class labels.

The vendor calendar incorrectly lists some NYSE holidays as sessions. The monthly stage
uses pinned `exchange_calendars` XNYS dates and records reconciliation differences.
It keeps the original vendor calendar; missing quotes never define market holidays.

In [ ]:
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None
monthly_run = run_monthly_sample(catalog, settings)
if monthly_run is None:
    display(Markdown("Monthly return construction is disabled in the settings file."))
else:
    # Preserve the loaded daily OHLCV/action/quality table. Build a separate monthly view.
    daily_df = monthly_run.sample.daily_prices.table.to_pandas().copy(deep=True)
    monthly_df = monthly_run.sample.panel.table.to_pandas().copy(deep=True)
    print("Daily source observations retained:", len(daily_df))
    print("Original-cohort monthly ledger rows:", len(monthly_df))
    show_table([asdict(w) for w in monthly_run.sample.panel.windows],
               ["outcome_month", "signal_date", "outcome_start", "outcome_end"])
    print(json.dumps(monthly_run.sample.panel.summary, indent=2))
    print("Measured cumulative source reads:")
    print(json.dumps(monthly_run.sample.loading_stats["total"], indent=2))
    print("Calendar reconciliation:")
    calendar_audit = monthly_run.sample.loading_provenance["calendar_reconciliation"]
    print(json.dumps({key: calendar_audit[key] for key in
                      ("policy", "package_version", "scope", "source_only_dates",
                       "independent_only_dates", "independent_session_count")}, indent=2))

### Read `monthly_df` before using its numbers

There is one row for **every source member at each signal cutoff**. `return_status`
separates valid ratios, unrequested observations, unresolved identities, missing
endpoints and rejected prices. A missing end can require terminal-event review; it
is not proof of a delisting and never deletes the constituent. The two-ID sample
does not establish complete price coverage of the intended research universe.

`daily_quote_path_complete` audits the independent XNYS daily sessions, independently of
`daily_backtest_ready`. Numeric month-end arithmetic can exist with an interior
gap; future daily backtests must resolve that path and cash-flow accounting.
Monthly volume sums are present only for complete valid daily observations, and
do not represent share turnover.

The `outcome_month` return becomes observable after its month closes. It must not
be a feature at the earlier `signal_date`. No next-month target ranks are built here.
Future feature construction must also retrieve an entrant's legitimate pre-entry
price history; membership selects prediction cohorts, not a truncation of price history.

In [ ]:
numerical_preview = None
if monthly_df is not None:
    numerical_preview = monthly_df.loc[monthly_df["return_status"] == "ok"].head(10)
    show_table(numerical_preview.to_dict("records"),
               ["source_security_id", "signal_date", "outcome_month", "outcome_end",
                "provider_adjusted_return", "return_status", "daily_quote_path_complete"])
    print("Engineering numerical preview only. Full monthly_df retains all", len(monthly_df), "cohort rows.")
    print("daily_df remains the separate daily-frequency source view for later backtesting.")

## 10. Keep daily and monthly tables independently

`daily_df` contains the original bounded daily quotes; `monthly_df` is a separate
derived cohort/return table. Monthly aggregation never overwrites the daily source.
The Arrow source table also remains unchanged. This sample is not the future full
daily execution dataset, and no backtest is run.

Persisted private evidence contains **`daily.parquet`**, **`monthly.parquet`** and
**`audit.json`** with source, code, request and table hashes, plus both original and
independent calendar tables. Identical reruns reuse
it; changed evidence under unchanged provenance is rejected. Clear notebook outputs
before committing; source observations and generated tables remain outside Git.

In [ ]:
monthly_artifact_path = None
if monthly_run is not None and settings["monthly_returns"]["persist"]:
    monthly_artifact_path = save_monthly_sample(ROOT, monthly_run)
    print("Private daily/monthly evidence:", monthly_artifact_path)
else:
    print("Monthly evidence persistence is disabled.")

## 11. From feature engineering to shared targets and models

Full 2010–2023 feature and target coverage is now audited, and the shared class,
preprocessing and first-pilot conventions are fixed in their separate protocols.
Historical identity knowledge time and economic terminal-event accounting retain
provisional limits. The later rolling 10/1 evaluation and backtesting remain separate work.

`training_slice(...)` serves the continuous-return interface.
For both classifiers, use `classification_training_slice(feature_panel, shared_targets,
train_start=..., train_end=..., label_observed_through=...)`. It checks canonical
next-month intervals and completed-label cutoffs, retains original denominators,
and returns X, y and an audit ledger separately. It never fits a model or drops
missing labels. Do not select prediction rows by future label availability.
Fit imputation/selection only on the explicitly selected supervised training mask;
use calendar-separated validation, never random stock-row splits across months.

Section 15 constructs only a bounded shared-target check from existing samples;
section 16 verifies the saved full-target audit. Section 17 verifies the completed
first classifier pilot's saved reports. Run All does not fit estimators or backtest.
See `docs/FEATURES.md`, `docs/TARGET_CLASSES_RET.md` and `docs/CLASSIFIER_PILOT.md`.


## 10. Measured coverage after moving initialization to 2010

This checks complete paper inputs across the original historical S&P 500/400
cohorts, not just the two-ID smoke sample. It separates blocked mappings,
missing signal quotes, invalid quotes and incomplete exact monthly lookbacks.
The 2010 outcome year uses December 2009–November 2010 signals. Older raw
history is retained for warmup; changing initialization does not truncate it.

Run All verifies the saved audit and displays it. The explicit endpoint audit
is a separate worker, not an automatic daily warehouse build, model fit or
backtest. Its coverage must not select predictors or fit missingness values for
earlier folds. See [the measured audit](../docs/INITIALIZATION_2010.md).


In [ ]:
from dmv2.start_coverage import read_saved_start_coverage
initialization_report = initialization_yearly_df = None
INITIALIZATION_EVIDENCE_DIRECTORY = ROOT / "outputs/initialization_coverage/2010-v3"
initialization_report = read_saved_start_coverage(catalog, INITIALIZATION_EVIDENCE_DIRECTORY,
    start_year=int(settings["research_start"][:4]), end_year=settings["training"]["last_test_year"])
initialization_yearly_df = pd.read_csv(INITIALIZATION_EVIDENCE_DIRECTORY / "yearly.csv")
print("Verified initialization audit:", INITIALIZATION_EVIDENCE_DIRECTORY)
display(initialization_yearly_df)
print("This is complete paper-input coverage; it does not certify all added predictors or economic return accounting.")


### Reusable preparation helpers, defined without fitting

The future full feature panel and separate outcome ledger must be supplied
explicitly. `fit_mask` uses completed training labels only. Prediction eligibility
uses only the fifteen signal-time paper inputs; missing held-out outcomes never
select prediction rows. Both returned ledgers retain every original key.
The transformed frame keeps non-admitted rows as missing; an estimator must
select the returned fit mask explicitly rather than dropping the original ledger.


In [ ]:
from dmv2.features import training_slice
from dmv2.preprocessing import fit_preprocessor
from dmv2.paper_features import PAPER_FEATURE_NAMES

def prepare_training_inputs(full_feature_panel, separate_outcomes, *, train_start, train_end, label_observed_through):
    train = training_slice(full_feature_panel, separate_outcomes,
        train_start=train_start, train_end=train_end, label_observed_through=label_observed_through)
    paper_qualified = train.X[list(PAPER_FEATURE_NAMES)].notna().all(axis=1)
    fit_mask = paper_qualified & train.ledger["target_available"]
    fitted = fit_preprocessor(train, fit_mask=fit_mask)
    prepared = fitted.transform(train.X, prediction_mask=paper_qualified)
    return train, fit_mask, fitted, prepared

def prepare_prediction_inputs(signal_X, fitted):
    prediction_mask = signal_X[list(PAPER_FEATURE_NAMES)].notna().all(axis=1)
    return fitted.transform(signal_X, prediction_mask=prediction_mask)

print("Preparation helpers loaded. No imputation fit, model or backtest executed.")


## 12. Full feature panel and missingness, 2010–2023

This is the separate, explicitly authorized full S&P500/S&P400 union build.
Daily data feed the technical/context calculations before exact monthly snapshots.
The full-run worker also preserves raw daily quotes separately through December2023
for later backtesting. Daily quotes and future outcomes never enter the predictor table.

Run All verifies and displays saved full-run artifacts; it never starts the full build.
The existing `daily_df`, `monthly_df` and bounded `features_df` remain separate sample views.
`full_feature_dataset` and `full_daily_dataset` below are lazy datasets. To inspect one
signal, call `read_feature_partition(full_feature_run, signal_date="2009-12-31", columns=["zMOM1"])`.

Coverage is descriptive. Do not use full-history missingness to fit imputation or select
features for earlier folds. Historical identity/knowledge-time and economic certification
remain unverified. See [the full-build contract](../docs/FULL_FEATURE_PANEL.md).

Added context uses 80% as primary (90% sensitivity), requiring valid original members per variable/date/cohort after exhaustive requests.
The paper15 and individual lookbacks stay unchanged. Daily benchmarks apply the gate every day.
See [the context contract](../docs/CONTEXT_AVAILABILITY.md).


In [ ]:
# Clear state first so failed verification cannot leave stale panel roles/results.
context_panel_roles = full_feature_run = full_feature_dataset = full_daily_dataset = None
full_feature_coverage_df = full_cohort_coverage_df = None
from dmv2.full_panel import read_feature_partition
from dmv2.context_panel_roles import read_context_panel_roles
import pyarrow.dataset as arrow_dataset

try:
    context_panel_roles = read_context_panel_roles(catalog)
except FileNotFoundError:
    print("No verified primary/sensitivity pair is available. Configure the approved saved run pins; Run All does not start a build.")
else:
    full_feature_run = context_panel_roles["primary"]
    full_feature_coverage_df = full_feature_run["tables"]["by_feature_period"].copy()
    full_cohort_coverage_df = full_feature_run["tables"]["by_cohort_year"].copy()
    full_summary = full_feature_run["audit"]["summary"]
    print("Verified 80% primary full feature panel:", {key: full_summary[key] for key in
        ("completed_signals", "original_stock_months", "registered_features",
         "candidate_predictors", "paper_qualified_stock_months", "paper_qualified_fraction")})
    context_report = full_feature_coverage_df.loc[
        full_feature_coverage_df.universe.eq("combined") &
        full_feature_coverage_df.population.eq("paper_qualified") &
        full_feature_coverage_df.model_eligible.eq(True) &
        full_feature_coverage_df.role.isin(["cross_sectional", "market_state", "relative_context"])]
    print("Primary context policy:", full_feature_run["audit"]["configuration"]["context_policy"])
    display(context_report.groupby(["period", "role"], sort=False).agg(
        eligible_names=("feature_name", "count"),
        names_ever_finite=("finite_rows", lambda values: int(values.gt(0).sum())),
        finite_values=("finite_rows", "sum"),
        possible_values=("rows", "sum")))
    print("Daily observations feed features; the coverage aggregator itself reads completed features.")
    display(full_cohort_coverage_df.loc[full_cohort_coverage_df.universe.eq("combined")])
    display(full_feature_run["tables"]["by_family_year"].loc[
        lambda f: f.universe.eq("combined") & f.population.eq("paper_qualified")])
    full_run_directory = full_feature_run["directory"]
    full_feature_dataset = arrow_dataset.dataset([
        str(full_run_directory / "years" / str(year) / "features.parquet")
        for year in range(2010, 2024)], format="parquet")
    full_daily_dataset = arrow_dataset.dataset([
        str(full_run_directory / "years" / str(year) / "daily_prices_raw.parquet")
        for year in range(2010, 2024)], format="parquet")
    print("Primary 80% features and raw daily quotes remain separate lazy datasets. No imputation, labels, model or backtest fitted.")


## 13. 90% sensitivity alongside the 80% primary panel

By user direction,80% is now primary and 90% is the sensitivity check. This cell
uses the same verified pair as the primary dataset above; Run All never builds either.
Their original artifact bytes, paper15, technical298, exact temporal histories,
historical membership and separate raw daily data are unchanged.

Saved comparison tables retain their historical **90% →80%** orientation: gains mean
additional values released at 80%, not gains under the current 90% sensitivity.
Shared finite values are identical. Coverage and cohort composition are descriptive,
including a separate2010–2015 comparison. No model performance selected the threshold,
and missingness may remain systematic. See [the promotion/comparison record](../docs/CONTEXT_SENSITIVITY_80.md).


In [ ]:
context_sensitivity_run = context_sensitivity_dataset = context_sensitivity_comparison_df = None
context_comparison = None
if context_panel_roles is None:
    print("No verified primary/sensitivity pair. Run All does not build it.")
else:
    context_sensitivity_run = context_panel_roles["sensitivity"]
    context_comparison = context_panel_roles["comparison"]
    comparison_summary = context_comparison["summary"]
    print("Historical comparison orientation:90% →80%; current roles:80% primary,90% sensitivity.")
    print({key: comparison_summary[key] for key in (
        "eligible_context_predictors", "gained_context_values_original",
        "lost_context_values_original", "changed_shared_context_values_original",
        "paper15_exact_unchanged", "technical298_exact_unchanged",
        "preserved_artifacts_byte_identical")})
    context_sensitivity_comparison_df = context_comparison["tables"]["family_coverage_comparison"].copy()
    display(context_sensitivity_comparison_df.loc[
        lambda f: f.population.eq("paper_qualified")])
    display(context_comparison["tables"]["cohort_composition_period"])
    sensitivity_directory = context_sensitivity_run["directory"]
    context_sensitivity_dataset = arrow_dataset.dataset([
        str(sensitivity_directory / "years" / str(year) / "features.parquet")
        for year in range(2010, 2024)], format="parquet")
    print("Sensitivity context policy:", context_sensitivity_run["audit"]["configuration"]["context_policy"])
    print("Primary 80% and sensitivity 90% datasets remain separate. No model or imputer fitted.")


## 14. Modelling sketch: paper-mechanism baseline and random forest

The user selected the historical S&P 500 + S&P 400 union. Compare XGBoost and
random forest classifiers on Paper15 and the expanded 715-name candidate bank,
using the same prediction population, target definitions and RET payoff layer.
The baseline is a declared U.S. rolling/no-size adaptation of the paper.

This section reads draft configuration and generates calendar metadata only.
It does not read market observations, build targets, fit preprocessing/models
or evaluate the outer test. Proposed settings and unresolved decisions are
shown explicitly in [the design note](../docs/MODEL_COMPARISON.md).

This is the original design snapshot. Section 17 reads the separately authorized first four-arm pilot under `configs/classifier_pilot.json`; draft readiness flags above do not describe whether that saved pilot has completed.


In [ ]:
# Clear all planning state before validation; this cell imports no estimator.
model_plan = model_arms_df = model_schedule = model_schedule_df = model_readiness = None
from dmv2.model_design import (
    load_model_plan, experiment_matrix, outcome_schedule, readiness_report,
    validate_outcome_schedule,
)
import pandas as pd

model_plan = load_model_plan(ROOT / "configs/model_comparison.proposed.json")
model_arms_df = pd.DataFrame(experiment_matrix(model_plan))
model_schedule = outcome_schedule(model_plan)
validate_outcome_schedule(model_schedule, model_plan)
model_schedule_df = pd.DataFrame(model_schedule)
model_readiness = readiness_report(model_plan)
display(model_arms_df)
display(model_schedule_df.groupby("fold_id", sort=False).first()[[
    "train_outcome_start", "train_outcome_end", "development_outcome_end",
    "validation_outcome_start", "validation_outcome_end",
    "prediction_outcome_month", "feature_month",
]])
display(model_schedule_df.head(2)[[
    "prediction_outcome_month", "feature_month", "train_last_feature_month",
    "classifier_label_cutoff", "RET_outcome_start_month", "RET_outcome_end_month",
]])
print(model_readiness)
print("Planning only: no market-data read, label build, fit, model score or backtest.")


## 15. Shared monthly class targets and RET workers

Both XGBoost and random forest use these same workers. Exact-equal returns share
a descending midrank class; missing outcomes stay unlabelled without changing
signal-time prediction eligibility. Forward outcome requests are checked
separately from past feature requests. RET requires all ten realized classes
in every one of the prior 120 completed calendar months.

This cell reuses the already loaded two-month/two-ID engineering sample. It
makes no extra raw-data request or full-history label build. Such a partial
sample cannot release cohort ranks or a ten-year payoff vector; withholding
is expected. Inspect the aggregate reasons below. The separate saved 2010–2023 full target and
payoff coverage audit is verified in section 16.

Use `prepare_classification_inputs` below for both probability estimators.
The older `prepare_training_inputs` helper uses continuous endpoint returns;
it is not the classifier target adapter. See [the exact shared contract](../docs/TARGET_CLASSES_RET.md).


In [ ]:
# Reset state before protocol validation or worker calls.
target_ret_protocol = shared_target_run = target_classes_df = target_diagnostics_df = ret_sample_payoffs = None
from dmv2.target_workflow import load_target_ret_protocol, run_shared_target_sample
from dmv2.classification import classification_training_slice
from dmv2.ret import build_ret_payoffs, score_ret
from dmv2.preprocessing import fit_preprocessor
from dmv2.paper_features import PAPER_FEATURE_NAMES

def prepare_classification_inputs(feature_panel, shared_targets, *, train_start, train_end,
                                  label_observed_through, columns=None):
    train = classification_training_slice(feature_panel, shared_targets,
        train_start=train_start, train_end=train_end,
        label_observed_through=label_observed_through, columns=columns)
    paper_qualified = train.X[list(PAPER_FEATURE_NAMES)].notna().all(axis=1)
    fit_mask = paper_qualified & train.ledger["target_available"]
    fitted = fit_preprocessor(train, fit_mask=fit_mask)
    prepared = fitted.transform(train.X, prediction_mask=paper_qualified)
    return train, fit_mask, fitted, prepared

target_ret_protocol = load_target_ret_protocol(ROOT)
if feature_run is None or monthly_run is None:
    print("No existing feature/monthly sample: target workers are loaded; no data build started.")
else:
    shared_target_run = run_shared_target_sample(
        feature_run.panel, monthly_run.sample.panel, target_ret_protocol)
    target_classes_df = shared_target_run.targets.table.to_pandas().copy(deep=True)
    target_diagnostics_df = shared_target_run.targets.diagnostics.to_pandas().copy(deep=True)
    ret_sample_payoffs = shared_target_run.payoffs
    print(shared_target_run.summary)
    display(target_diagnostics_df)
    print("Same shared class and RET interfaces for XGBoost and random forest.")
    print("Missing labels retain prediction rows; daily/feature tables remain separate and unchanged.")
    print("No preprocessing fit, model fit, full-history target build or backtest executed.")


## 16. Full target and RET coverage, 2010–2023

This reads the saved, explicitly authorized full target build and coverage reports.
All original historical membership keys and the same Paper15 prediction cohorts
are retained for both models. Targets and outcomes remain separate from the
80% primary / 90% sensitivity feature datasets and from raw daily data.

The reports show all 168 months, missing outcomes, ten realized classes and the
120-calendar-month RET grids. The nominal 2020–2023 tests have 48 monthly signals.
Earlier RET vectors may be withheld because the outcome history starts in 2010.
Coverage describes observed outcomes under our provisional identity policy;
it does not certify historical knowledge-time identity or terminal cash flows.

Run All verifies saved fingerprints and displays small reports only. The full
label and monthly-outcome views below are lazy; `read_full_targets(full_target_run)`
explicitly loads the shared class panel when a later training step needs it.
No full build, preprocessing fit, classifier fit or backtest starts here.
See [the full target audit](../docs/FULL_TARGET_PANEL.md).


In [ ]:
# Clear state before verification; a failed rerun must not show stale coverage.
full_target_run = full_target_dataset = full_monthly_outcome_dataset = None
full_target_coverage_df = full_target_class_coverage_df = full_ret_coverage_df = None
from dmv2.full_target_panel import read_full_target_run, read_full_targets
import pyarrow.dataset as target_arrow_dataset

try:
    full_target_run = read_full_target_run(catalog)
except FileNotFoundError:
    print("No completed full target audit is available. Run All does not start a build.")
else:
    full_target_coverage_df = full_target_run["tables"]["target_coverage_year"].copy()
    full_target_class_coverage_df = full_target_run["tables"]["target_class_year"].copy()
    full_ret_coverage_df = full_target_run["tables"]["ret_coverage_signal"].copy()
    target_summary = full_target_run["audit"]["summary"]
    print("Verified full target coverage:", {key: target_summary["targets"][key] for key in
        ("calendar_months", "original_members", "prediction_eligible_members", "labelled_members",
         "labelled_prediction_coverage", "eligible_unlabelled_members", "months_with_empty_classes")})
    display(full_target_coverage_df[["outcome_year", "original_members",
        "prediction_eligible_members", "labelled_members", "labelled_prediction_coverage",
        "eligible_unlabelled_members", "months_with_empty_classes"]])
    display(full_target_class_coverage_df.pivot(index="outcome_year", columns="class_id", values="labelled_members"))
    display(full_target_run["tables"]["rolling_training_coverage"])
    print("RET coverage:", target_summary["ret"])
    display(full_target_run["tables"]["ret_coverage_year"])
    print("Eligible missing-return reasons:", target_summary["targets"]["eligible_return_observation_status_counts"])
    directory = full_target_run["directory"]
    full_target_dataset = target_arrow_dataset.dataset(str(directory / "targets.parquet"), format="parquet")
    full_monthly_outcome_dataset = target_arrow_dataset.dataset(str(directory / "monthly_returns.parquet"), format="parquet")
    print("Targets, monthly outcomes, features and daily data stay separate. No fit or backtest executed.")


## 17. First classifier pilot: train 2010–2017, validate 2018–2019

The user selected all four arms: RF and XGBoost, each on Paper15 and the expanded
715-name candidate bank. This is the first chronological development pilot;
the later rolling ten-year outer tests remain untouched by model evaluation.
The 80% context panel remains primary. Daily data stay separate and unchanged.

Each arm uses one fixed-seed fit on the same 77,690 labelled training rows.
Training-only medians, missing indicators and all-missing/constant exclusions
are shared by estimators using the same feature set. Every arm predicts all
20,755 eligible validation rows, including the 46 without an observed label.
Only the common observed-label mask contributes to validation metrics.

Primary comparison: **equal-month mean log loss** (lower is better); secondary
Brier sums ten squared class errors, plus accuracy and calibration diagnostics.
Uniform and training-only class-prior forecasts are reference predictions.
No tuning, early stopping, calibration fitting, outer refit or backtest occurs.
RET is withheld in this early validation period under the strict 120-month rule.

The cell below verifies saved files and displays small reports. **Run All never
starts or repeats a fit**, loads a pickled model, or rebuilds prepared matrices.
Use the explicit CLI from [the frozen pilot record](../docs/CLASSIFIER_PILOT.md).


In [ ]:
# A stale or failed rerun cannot leave earlier pilot results visible.
classifier_pilot_run = classifier_pilot_comparison_df = None
from dmv2.classifier_pilot import read_classifier_pilot_run

try:
    classifier_pilot_run = read_classifier_pilot_run(catalog)
except FileNotFoundError:
    print("No completed classifier pilot is saved. Run All does not start fitting.")
else:
    classifier_pilot_comparison_df = classifier_pilot_run["tables"]["comparison"].copy()
    columns = ["arm_id", "input_predictors", "retained_predictors", "matrix_width",
               "prediction_rows", "labelled_rows", "mean_monthly_log_loss",
               "mean_monthly_brier_sum10", "mean_monthly_accuracy", "pooled_top_label_ece"]
    display(classifier_pilot_comparison_df[columns])
    print("Verified pilot:", classifier_pilot_run["audit"]["summary"])
    print("Training outcomes: 2010–2017. Validation outcomes: 2018–2019.")
    print("Same cohorts and observed-label mask across all four arms. Daily data stay separate.")
    print("Development diagnostics only: no 2020–2023 model evaluation, RET or backtest.")


## 18. Daily development backtest: 2018–2019, fixed ten per side

The same saved four model arms feed **separate class 1/10 and 2/9 portfolios**.
Each direction selects ten highest corresponding probabilities **within its mode class**;
equal weights allocate 50% long / 50% short. This is a V2 breadth adaptation.
RET stays withheld under its unchanged 120-month history gate. No 2020–2023
financial evaluation or model refit is part of this stage.

Daily adjusted-price proxy accounting holds units through next-open monthly
rebalancing. The separate **10% per-asset stop** variant leaves stopped names in
cash; gap fills use the open and intraday crossings assume an idealized threshold fill.
Cost sensitivities are 0/5/10/25 bp per one-way traded notional. Missing required
valuations withhold full-period P&L while preserving positions and coverage.
These results retain provisional identity and economic-accounting limitations.

The cell below **only verifies saved reports**. To run the bounded backtest
explicitly, use [the daily-backtest worker and protocol](../docs/DAILY_BACKTEST.md).
Daily source data and the earlier `daily_df`, `monthly_df` and `features_df` stay separate.


In [ ]:
# Clear old state before checking the immutable saved development backtest.
daily_backtest_run = daily_backtest_comparison_df = daily_backtest_selection_df = None
daily_backtest_ret_readiness_df = daily_backtest_failures_df = None
from dmv2.daily_backtest import read_daily_backtest

try:
    daily_backtest_run = read_daily_backtest(catalog)
except FileNotFoundError:
    print("No saved daily development backtest. Run All does not start one.")
else:
    daily_backtest_comparison_df = daily_backtest_run.comparison.copy()
    daily_backtest_selection_df = daily_backtest_run.selection_coverage.copy()
    daily_backtest_ret_readiness_df = daily_backtest_run.ret_readiness.copy()
    daily_backtest_failures_df = daily_backtest_run.failures.copy()
    columns = ["arm", "basket", "variant", "cost_bps", "status", "total_return",
               "sharpe_zero_cash_rate", "maximum_drawdown", "stop_count",
               "valid_close_sessions", "first_failure_date"]
    display(daily_backtest_comparison_df.loc[daily_backtest_comparison_df.cost_bps.eq(0), columns])
    incomplete = int((~daily_backtest_comparison_df.complete_proxy_path).sum())
    print(f"{incomplete}/{len(daily_backtest_comparison_df)} paths lack complete daily valuations; full-period performance is withheld for those paths.")
    display(daily_backtest_ret_readiness_df[["outcome_month", "status"]])
    if not daily_backtest_failures_df.empty:
        display(daily_backtest_failures_df)
    print("Verified saved daily reports:", daily_backtest_run.path)
    print("10 longs + 10 shorts; 50% / 50%. Class 1/10 and 2/9 remain separate.")
    print("RET withheld: incomplete prior 120 months. No 2020–2023 evaluation or model refit.")
    print("Provider-adjusted proxy results; identities, terminal cash flows and executable costs unverified.")


## 19. Corporate-action and reviewed-bar rerun, 2018–2019

This additive run preserves the eight original selected/rebalance files byte for
byte. Verified cash and stock entitlements apply to existing holdings; an entry
after a security has ceased trading stays unfilled with its allocation in cash.
The original three rejected OHLC bars require independent, compatible source
evidence. Source bars remain unchanged; corrections and their evidence are separate.

The 50% long / 50% short, ten names per direction, class 1/10 and 2/9 baskets,
10% per-asset stop alternative and cost sensitivities are unchanged. Unquoted
successors, halts and missing held valuations still withhold the affected path's
full-period performance. Complete paths alone must not become a model leaderboard.
RET remains withheld. No model fitting or 2020–2023 evaluation occurs here.

The cell only verifies saved hashes and small reports. It never starts accounting
or fetches prices. See [the action rerun protocol](../docs/ACTION_BACKTEST.md) and
[primary event evidence](../docs/CORPORATE_ACTION_EVIDENCE.md).

The known historical BBBY-to-Overstock route conflict also affects upstream inputs.
This stage displays coverage, not a strategy-performance ranking. See [the rerun results and limitations](../docs/ACTION_BACKTEST_RESULTS.md).


In [ ]:
# Clear state before checking the saved additive corporate-action rerun.
action_backtest_run = action_backtest_comparison_df = action_backtest_repairs_df = None
action_backtest_failures_df = action_backtest_actions_df = None
from dmv2.action_backtest import read_action_backtest

try:
    action_backtest_run = read_action_backtest(catalog)
except FileNotFoundError:
    print("No saved corporate-action rerun. Run All does not start one.")
else:
    action_backtest_comparison_df = action_backtest_run.comparison.copy()
    action_backtest_repairs_df = action_backtest_run.ohlc_repairs.copy()
    action_backtest_failures_df = action_backtest_run.failures.copy()
    action_backtest_actions_df = action_backtest_run.action_coverage.copy()
    print("Known historical BBBY-to-Overstock identity conflict: strategy-performance interpretation is withheld.")
    print("See docs/ACTION_BACKTEST_RESULTS.md for direct and upstream exposure.")
    columns = ["arm", "basket", "variant", "cost_bps", "status",
               "valid_close_sessions", "first_failure_date", "action_count"]
    display(action_backtest_comparison_df.loc[action_backtest_comparison_df.cost_bps.eq(0), columns])
    print("Reviewed bar repairs:", len(action_backtest_repairs_df))
    incomplete = int((~action_backtest_comparison_df.complete_proxy_path).sum())
    print(f"{incomplete}/{len(action_backtest_comparison_df)} paths remain incomplete; their full-period performance is withheld.")
    if not action_backtest_failures_df.empty:
        display(action_backtest_failures_df)
    display(action_backtest_actions_df)
    print("Verified action rerun:", action_backtest_run.path)
    print("Original selections, rebalances and daily source files preserved. No 2020–2023 evaluation.")
    print("Provisional identities and adjusted-price proxy; actual settlement, borrow and fill realism remain unverified.")


## 20. BBBY identity amendment and rebuild dependencies

Old Bed Bath & Beyond (`US0758961009`) was routed to a historical Overstock
native security. The separate versioned amendment disables this known wrong
route and retains every original cohort row and review flag. **A verified
replacement native security and its price history are still unresolved.**
Blocking a wrong route does not repair the historical prices or certify a
model-ready panel. The dependency review identifies calculations requiring
recomputation; its counts are not measured changes to feature or label values.
The original feature, target, pilot and backtest runs remain reference evidence.

The cell reads verified saved metadata reports only. It does not decode feature,
target or price panels, change the active catalog, build corrected data, fit a
model, rerun a backtest or evaluate 2020–2023 financial values. See
[the amendment and rebuild scope](../docs/BBBY_IDENTITY_AMENDMENT.md).


In [ ]:
# Clear this review's state before verification; original runs stay unchanged.
bbby_identity_review = bbby_identity_summary = None
from dmv2.identity_amendment import read_saved_identity_review

try:
    bbby_identity_review = read_saved_identity_review(
        ROOT / "data/derived/identity_amendments/old-bbby-native-conflict-20261016-v1")
except FileNotFoundError:
    print("No saved BBBY identity review. Run All does not build it or fetch prices.")
else:
    bbby_identity_summary = bbby_identity_review["summary"]
    print(json.dumps(bbby_identity_summary, indent=2, sort_keys=True))
    print("Verified identity review:", bbby_identity_review["path"])
    print("Known wrong route disabled; replacement native ID and price history still unresolved.")
    print("This identity-review snapshot predates the rebuild. Section 21 shows the separate data-limited run.")


## 21. Data-limited 2018–2019 reconstruction — saved reports only

The latest user direction permits a separate reconstruction without waiting for
replacement sources. BBBY's known wrong route remains missing; Dean Foods and
Rowan stay in valid training data and predictions but are excluded from trading
before top-ten selection throughout this development experiment. These are
retrospective data-quality restrictions, so the results are conditional on the
restricted universe and do not remove selection bias.

Earlier sections preserve the original reference runs. This section reads only
the explicitly pinned new run: annual summaries, four-fit model reports and the
64-scenario execution report when saved. It never rebuilds, compiles quotes,
loads a feature/price/target panel, deserializes models or starts a backtest.
Default verification checks small saved receipts; expensive full-source byte
hashing is an explicit optional worker argument. Integrity errors raise.
The run remains 2010–2017 training and 24 prediction months in 2018–2019;
2020–2023 evaluation has not started.

See [the frozen fallback method](../docs/DATA_LIMITED_BACKTEST.md).


In [ ]:
# Clear prior views before every saved-report check.
fallback_saved_report = None
fallback_year_summary = fallback_model_comparison = fallback_backtest_comparison = None
from dmv2.fallback_reporting import read_data_limited_run
from IPython.display import display

FALLBACK_RUN = ROOT / "outputs/data_limited_backtest/2010-2019-50aa6e09f11b687c"
FALLBACK_IDENTITY = "sha256:50aa6e09f11b687c36cd3cf52ed8eb51fdde591fb2391cd58162d02edbb29b10"
# Final verified audit receipts for this completed numerical run.
FALLBACK_EXPECTED_RECEIPTS = {'backtest': 'sha256:d49cf2ca11e4e572c19fe7fe0599d0b7e77b906bf4c80199bdc7adac1c5905e3', 'features': 'sha256:79b862fb544c1b8b24b590f07b86f6abb658c4849cca9320840495a034d8aa47', 'models': 'sha256:8697cd9feccedaf07042a5bf700282b22304066e15f6edc99ca09eee65ee951e'}
try:
    fallback_saved_report = read_data_limited_run(
        ROOT, FALLBACK_RUN, identity_sha256=FALLBACK_IDENTITY,
        expected_receipts=FALLBACK_EXPECTED_RECEIPTS, verify_sources=False)
except FileNotFoundError:
    print("The new saved run is not available yet. Run All starts no computation.")
else:
    print(json.dumps(fallback_saved_report["summary"], indent=2, sort_keys=True))
    fallback_year_summary = fallback_saved_report["year_summaries"]
    fallback_model_comparison = fallback_saved_report["model_comparison"]
    fallback_backtest_comparison = fallback_saved_report["backtest_comparison"]
    if not fallback_year_summary.empty:
        display(fallback_year_summary)
    if not fallback_model_comparison.empty:
        display(fallback_model_comparison)
    if not fallback_backtest_comparison.empty:
        display(fallback_backtest_comparison)
    print("Data-limited restricted universe; original runs preserved; no 2020–2023 evaluation.")


## 22. Independent quote repair and complete 2018–2019 backtest — saved reports only

This additive run repairs independent-source daily quotes and the SNI cash/DISCK
entitlement plus VVC and TSS retirements, then reuses the four exact 2010–2017 fits. It rebuilds only the
affected 2018–2019 inputs and forecasts. The user selected balanced breadth:
N=min(10, long-mode candidates, short-mode candidates), with equal counts on both
sides and 50%/50% capital. Nine per side therefore means 5.56% per position.

The complete evaluation is **24 months, 503 sessions, January 2018–December 2019**.
Total return below spans both years; volatility and Sharpe are annualized.
Earlier sections remain reference runs. This cell reads bounded saved receipts
and reports only. Run All starts no fetch, build, model load, fit or backtest.
The fixed retrospective DF/RDC restriction and unresolved BBBY/provisional
identity limitations remain; 2020–2023 evaluation stays untouched. See
[the method](../docs/INDEPENDENT_QUOTE_BACKTEST.md).


In [ ]:
# Clear prior views; verify explicitly pinned saved reports only.
independent_saved_report = independent_backtest_comparison = None
independent_model_comparison = independent_year_summary = None
from dmv2.completed_reporting import read_completed_run

INDEPENDENT_RUN = ROOT / 'outputs/independent_backtest/2018-2019-eb3cd1e297166a2b'
INDEPENDENT_IDENTITY = 'sha256:eb3cd1e297166a2b842f666d2c54d9226b6a31f283b1e21c6667d8361b19f7d9'
INDEPENDENT_RECEIPTS = {'features': 'sha256:1e75f1d72a27bacf021cba864f8a49d2301dde06927e07bbb329f3990e180408', 'models': 'sha256:9e52a3374ef9cea259c9a1252d1b4315df105819aa5932308c0336dc93d8023c', 'backtest': 'sha256:c8d05ed09d31cb750d85f5c4de5caa377639076e34670efd645dba0ec55a6c57', 'original_backtest': 'sha256:992ce549dbab0661fdda41e9aad347ce270f82c419570239cd432a23d1510563'}
try:
    independent_saved_report = read_completed_run(
        ROOT, INDEPENDENT_RUN, identity_sha256=INDEPENDENT_IDENTITY,
        expected_receipts=INDEPENDENT_RECEIPTS, verify_sources=False)
except FileNotFoundError:
    print("Saved independent repair is unavailable. Run All starts no computation.")
else:
    independent_year_summary = independent_saved_report["year_summaries"]
    independent_model_comparison = independent_saved_report["model_comparison"]
    independent_backtest_comparison = independent_saved_report["backtest_comparison"]
    print("24 months / 503 sessions: January 2, 2018–December 31, 2019.")
    print("Four exact prior fits reused; zero new fits. 2020–2023 untouched.")
    display(independent_year_summary)
    shown = independent_backtest_comparison.loc[
        independent_backtest_comparison.cost_bps.eq(10),
        ["arm", "basket", "variant", "cost_bps", "total_return", "cagr",
         "annualized_volatility", "maximum_drawdown", "sharpe_zero_cash_rate"]].copy()
    for name in ("total_return", "cagr", "annualized_volatility", "maximum_drawdown"):
        shown[name] = shown[name].map(lambda value: f"{value:.2%}")
    shown = shown.rename(columns={"total_return": "total_return_2018_2019"})
    display(shown)
    print("Basket N=min(10, long candidates, short candidates), equal counts; 50%/50%.")
    print("Data-limited restricted universe; provisional identities and execution accounting.")
    print("Full report:", INDEPENDENT_RUN / "reports/RESULTS.md")


## 23. Annual rolling walk-forward, 2020–2023 — verified saved reports

Each year has four fixed model arms (RF/XGBoost × Paper15/expanded), trained on only the preceding ten completed outcome years. Training windows are 2010–2019 → 2020; 2011–2020 → 2021; 2012–2021 → 2022; and 2013–2022 → 2023. Predictions remain monthly. Sixteen estimator fits and eight training-only preprocessing states are shared across the same original PIT admission and observed-label masks.

The daily accounting ledger runs continuously across all 48 months and 1,006 XNYS sessions, January 2, 2020–December 29, 2023. Annual retraining adds no extra liquidation. Class 1/10 and 2/9 portfolios remain separate, with balanced N=min(10,long candidates,short candidates), intended 50% long/50% short, next-open execution, unchanged unstopped and 10% per-asset-stop variants and explicit 0/5/10/25 bps costs. Stopped positions and unfilled retired-security orders remain cash without replacement. RET is not evaluated in this class-basket experiment.

This section reads only small, saved reports after exact feature/model/backtest audits and all three independent verification receipts pass. It never builds panels, fits a model, deserializes an estimator, reads source prices or opens prepared matrices. Missing or withheld full-period results remain unavailable; they are not replaced by prefix performance.

`walkforward_monthly_10bps` contains portfolio returns and NAV, not modelling features or input price candles. Existing daily and monthly input dataframes remain separate and unchanged. Daily history remains necessary for feature calculation and later execution audits. Provisional identities, the fixed DF/RDC execution restrictions, source coverage and corporate-action proxy conventions remain research limitations.


In [ ]:
# Clear every new view before checking receipts, so stale results cannot survive a failure.
walkforward_saved_report = walkforward_model_comparison = None
walkforward_backtest_comparison = walkforward_yearly = None
walkforward_class_metrics = walkforward_monthly_10bps = None
walkforward_selection_coverage = walkforward_year_summary = None
from dmv2.walkforward_reporting import read_walkforward_run

# Fill these literal values only after all independent verification completes.
WALKFORWARD_RUN = ROOT / "outputs/walkforward/2020-2023-aacd7579fcf21e6a"
WALKFORWARD_IDENTITY = "sha256:aacd7579fcf21e6ad1e6d94d0c5af2d8cb510ff7af31e87c4bebe69621a5fca2"
WALKFORWARD_RECEIPTS = {
    "features": "sha256:63b21801b8a39d2b7d16580b5ddd13c8badd99155abac716f06860cf9b8506a8",
    "models": "sha256:c4ab12bfaf9f559772012219842ea03abe42908a1ec91d09f44a1f2de8d3e19f",
    "backtest": "sha256:201dcf7f5622790d519b61a940a8017e107c5c54f00f2ada8f963cfcb4554f90",
    "feature_verification": "sha256:61078970a3929b53896b590da31b4fac9d3fa5bc6068c3c0e0a8b96733e85c26",
    "model_verification": "sha256:c9a9cf96842633f19c8c0b11ad37815af2e14b54c50b9393b3093f675dd3ecdd",
    "backtest_verification": "sha256:b1c4d6e5752a082ec516503302633309951567a39b5e2a3e907f523fb1850af7",
}
if "__PENDING" in WALKFORWARD_IDENTITY or any("__PENDING" in value for value in WALKFORWARD_RECEIPTS.values()):
    print("Awaiting completed walk-forward outputs and all independent receipts. Run All starts no computation.")
else:
    try:
        walkforward_saved_report = read_walkforward_run(
            ROOT, WALKFORWARD_RUN, identity_sha256=WALKFORWARD_IDENTITY,
            expected_receipts=WALKFORWARD_RECEIPTS, verify_sources=False)
    except FileNotFoundError:
        print("Saved walk-forward results or verification receipts are unavailable. Run All starts no computation.")
    else:
        walkforward_year_summary = walkforward_saved_report["year_summaries"]
        walkforward_model_comparison = walkforward_saved_report["model_comparison"]
        walkforward_backtest_comparison = walkforward_saved_report["backtest_comparison"]
        walkforward_yearly = walkforward_saved_report["backtest_yearly"]
        walkforward_class_metrics = walkforward_saved_report["model_classes_1_2_9_10"]
        walkforward_monthly_10bps = walkforward_saved_report["monthly_10bps"]
        walkforward_selection_coverage = walkforward_saved_report["selection_coverage"]
        status = walkforward_saved_report["backtest_summary"]
        print("48 monthly predictions / 1,006 daily sessions: January 2, 2020–December 29, 2023.")
        print("16 annual estimator fits; 8 training-only preprocessing states; continuous NAV across years.")
        print(f"Complete proxy paths: {status['complete_proxy_paths']}/64; withheld paths: {status['failed_closed_paths']}/64.")
        display(walkforward_model_comparison)
        if not walkforward_class_metrics.empty:
            display(walkforward_class_metrics.loc[walkforward_class_metrics.arm_id.isin(
                ["rf_paper15", "xgb_paper15", "rf_extended", "xgb_extended"])])
        shown = walkforward_backtest_comparison.loc[
            walkforward_backtest_comparison.cost_bps.eq(10),
            ["arm", "basket", "variant", "status", "total_return", "cagr",
             "annualized_volatility", "maximum_drawdown", "sharpe_zero_cash_rate"]].copy()
        display(shown.rename(columns={"total_return": "total_return_2020_2023"}))
        display(walkforward_year_summary)
        print("Source prices, daily data, feature panels and monthly targets were not opened by this section.")
        print("Data-limited restricted universe; provisional identity and accounting conventions remain.")


## 24. Corrected-source walk-forward, 2020–2023 — current saved results

Section 23 remains the original-source reference. This separate replay uses the reviewed El Paso/Empire route correction and OPCH daily-bar repair, refits all sixteen estimators under the same recipe, and repeats the unchanged daily accounting. The earlier source-repair handoff describes why those original models became stale; it does not invalidate this separately verified new run.

Training remains rolling ten completed years, with annual refits and monthly predictions. RF expanded with the 10% per-asset stop and class1/10 basket is the primary focus. Class2/9 remains separate. Balanced up-to-ten names per side, monthly 50%/50% initial allocation, next-open execution, costs and stop rules are unchanged. Stops leave cash until the next monthly decision. The continuous horizon is 2 January 2020–29 December 2023: 48 months and 1,006 sessions.

This cell checks exact saved input/model/accounting audits and independent receipts, then reads two small report tables. It starts no fit or backtest, reloads no model and reads no source prices or prepared matrices. Feature verification here means authenticated reuse of the reviewed inputs and their prior bounded formula proofs, not a new exhaustive certification of all715 formulas. Historical reconstructed identity, coverage and execution assumptions remain provisional. Withheld full-period metrics remain missing.


In [ ]:
# Clear views before checking exact receipts; a failure cannot retain stale results.
corrected_walkforward_saved = corrected_walkforward_comparison = None
corrected_walkforward_yearly = corrected_rf_expanded_sl = None
corrected_rf_expanded_sl_yearly = None
from dmv2.source_corrected_reporting import read_source_corrected_reports

CORRECTED_WALKFORWARD_BINDINGS = ROOT / "outputs/walkforward_source_corrected/2020-2023-20261016-v1/report_bindings.json"
CORRECTED_WALKFORWARD_BINDINGS_SHA = "sha256:a078830ad9c4e28faf915be924e1ab610f443905c039b4b401cb206e3f2b0aac"
CORRECTED_WALKFORWARD_IDENTITY = "sha256:559ba2f7fbe187f192b40af2ab8fbe0a4fd70a1b843e2cbd0503f7e35729fcae"
try:
    corrected_walkforward_saved = read_source_corrected_reports(
        CORRECTED_WALKFORWARD_BINDINGS, root=ROOT,
        expected_bindings_sha256=CORRECTED_WALKFORWARD_BINDINGS_SHA,
        identity_sha256=CORRECTED_WALKFORWARD_IDENTITY)
except FileNotFoundError:
    print("Verified corrected-run files are unavailable. No computation was started.")
else:
    corrected_walkforward_comparison = corrected_walkforward_saved["comparison"]
    corrected_walkforward_yearly = corrected_walkforward_saved["yearly"]
    corrected_rf_expanded_sl = corrected_walkforward_saved["rf_expanded_sl"]
    corrected_rf_expanded_sl_yearly = corrected_walkforward_saved["rf_expanded_sl_yearly"]
    corrected_status = corrected_walkforward_saved["audit"]
    print("Corrected inputs; 16 refits / 8 train-only preprocessing states; 48 months / 1,006 sessions.")
    print(f"Complete proxy paths: {corrected_status['complete_proxy_paths']}/64; withheld: {corrected_status['failed_closed_paths']}/64.")
    print("Primary: RF expanded / class1–10 / 10% per-asset stop; gross and 10bps shown separately.")
    display(corrected_rf_expanded_sl)
    display(corrected_rf_expanded_sl_yearly)
    corrected_columns = ["arm", "basket", "variant", "status", "total_return", "cagr",
                         "maximum_drawdown", "annualized_volatility", "sharpe_zero_cash_rate"]
    display(corrected_walkforward_comparison.loc[
        corrected_walkforward_comparison.cost_bps.eq(10), corrected_columns])
    print("Total return covers 2020–2023. CAGR/volatility/Sharpe are annualized; daily-close drawdown.")
    print("10bps is per one-way traded notional. Borrow/funding remain zero; proxy fills remain provisional.")
